# MAI-IHLT
## Lab Blocks - Lexical Level: Spelling Corrector
#### Authors:
- Josep de Cid (josep.de.cid@est.fib.upc.edu)
- Gonzalo Recio (gonzalo.recio@est.fib.upc.edu)

## Basic Approach

Using the **Levenshtein edit distance** (number of characters that need to be substituted, inserted, or deleted, to transform the a word into another one).

In [1]:
english_words = []
with open('wordsEn.txt', mode='r') as f:
    english_words = ''.join(f.readlines()).split('\n')

In [2]:
from time import time
from nltk.metrics.distance import edit_distance

def timeit(f):
    def timed(*args, **kw):
        ts = time()
        result = f(*args, **kw)
        te = time()
        print(f'{f.__name__} with "{args[0] if len(args) == 1 else args[1]}" took {te - ts:.6f}s')
        return result
    return timed

@timeit
def correct_word(word):
    if word in english_words:
        print(f'Word "{word}" is correct!')
    else:
        distances = [(x, edit_distance(x, word)) for x in english_words]
        min_distance = min(distances, key=lambda k: k[1])[1]
        
        filtered_min_distances = filter(lambda x: x[1] == min_distance, distances)
        suggenstion_words = list(map(lambda x: x[0], filtered_min_distances))
        
        print(f'Word "{word}" is invalid. Did you mean:')
        for suggestion in suggenstion_words:
            print(f' - {suggestion}')

In [3]:
correct_word('something')

Word "something" is correct!
correct_word with "something" took 0.002155s


In [4]:
correct_word('soemthing')

Word "soemthing" is invalid. Did you mean:
 - seething
 - sheathing
 - sleuthing
 - something
 - soothing
 - southing
correct_word with "soemthing" took 10.373993s


In [5]:
correct_word('actop')

Word "actop" is invalid. Did you mean:
 - actor
 - atop
correct_word with "actop" took 6.856850s


The main drawbacks of this method is the cost of calculating the Levenshtein edit distance for each word in the list, as its complexity is *O(n^2)* where *n* is the length of the word.

As we can see in the last 2 examples it takes around 10 seconds just to suggest the correction word. Hence, we must found a more suitable approach for real environments.

## Probabilistic Approach

Following [Peter Norvig's article](http://www.norvig.com/spell-correct.html), we will build a simple probabilistic spell corrector. The main idea is to find a correction _c_ such that *P(c|w)* is maximized. With Bayes' Theorem, we can simplify it into maximizing *P(c)P(w|c)*.

We model the code for this in a class that contains the 4 basic parts:
- Selection mechanism to choose the candidate with highest combines probability.
- Candidate Model to select a subset of cadidates as possible corrections.
- Language model as the probability of a word *w* to appear in a text
- Error model as the probability of *w* being typed when the author meant _c_.

Instead of using a dictionary, which will give us probabilities of 1/N for each word, we use a Big *.txt* file from _The Adventures of Sherlock Holmes by Sir Arthur Conan Doyle_ as a real world corpus, combined with the previously used dictionary, to avoid missing any word.

The weighting method for recommending words is not only based on probabilities but also on distances. The more edit steps we perform, the higher the number of initial candidates grow, when much of them isn't going to be a feasible word and will be discarded later. The growth branching factor is *54n+25*, being _n_ the length of the word.

Also it's not a desirable behavior than, typing "helo", the system recommends "bye" if *e.g.* there are no more closer words, even if this one is super far away. Hence, we create a parammeter *d* which limits the maximum number of edit steps from the original word. This one is used in the weighting method to penalize words that need more steps to be reached, combined with their probabilities.

In [ ]:
# Uncomment and run this cell to obtain the big file
# !wget http://www.norvig.com/big.txt

In [6]:
import re
import string
from functools import reduce
from collections import Counter


class SpellCorrector:
    def __init__(self, words_file_paths: list, gamma=1):
        """Read a dictionary of words and built the language model"""
        words = []
        for path in words_file_paths:
            with open(path, mode='r') as f:
                words += re.findall(r'\w+', f.read().lower())

        self.words = Counter(words)
        self.n = sum(self.words.values())
        self.gamma = gamma

    @timeit
    def correct_word(self, word: str, r=5, d=1):
        """Correct word, showing 'r' results at distance 'd'"""
        if word in self.words:
            print(f'Word "{word}" is correct!')
        else:
            candidates = self.__candidates(word, d)
            if len(candidates) > 0:
                # Sort candidates by probability
                candidates.sort(key=self.__word_prob_score, reverse=True)
                print(f'Word "{word}" is invalid. Did you mean:')
                
                i, shown_candidates = 0, []
                for candidate in candidates:
                    if candidate[0] not in shown_candidates:
                        shown_candidates.append(candidate[0])
                        print(f' - {candidate[0]}')
                        i += 1
                    # Show at most r candidates
                    if i == r:
                        return
            else:
                print(f'No correction found for "{word}"')
            
    def __word_prob_score(self, word):
        """Calculate probability score of the given as percent of occurrences"""
        # Penalize higher distance edits reducing the score by a factor of gamma*distance
        return self.words[word[0]] / (self.n * (self.gamma * word[1]))
    
    def __candidates(self, word: str, d: int):
        """Generate candidates for 'word' at distance 'd'"""
        candidates = []
        candidates_d = [word]
        for i in range(1, d + 1):
            candidates_d = [list(self.__edits(candidate)) for candidate in candidates_d]
            candidates_d = list(reduce(lambda acc, x: acc + x, candidates_d, []))
            
            candidates_d_index = list(map(lambda c: (c, i), candidates_d))
            candidates += candidates_d_index
        
        candidates = self.__known_words(candidates)
        
        return candidates
    
    def __known_words(self, words: str) -> list:
        """Filter words only among set of existing words"""
        return list(filter(lambda w: w[0] in self.words, set(words)))
    
    def __edits(self, word: str) -> list:
        """Generate the set of editable words from 'word' at distance 'd'"""
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        edited_words = []
        for w1, w2 in splits:
            for c in string.ascii_lowercase:
                # Inserts
                edited_words.append(w1 + c + w2)

                # Replaces
                edited_words.append(w1 + c + w2[1:])

            # Transposes
            if len(w2) >= 2:
                edited_words.append(w1 + w2[1] + w2[0] + w2[2:])

            # Deletes
            edited_words.append(w1 + w2[1:])

        return set(edited_words)

corrector = SpellCorrector(['big.txt', 'wordsEn.txt'])

In [7]:
corrector.correct_word('soemthing', d=1)

Word "soemthing" is invalid. Did you mean:
 - something
correct_word with "soemthing" took 0.000992s


In [8]:
corrector.correct_word('soemthing', d=2)

Word "soemthing" is invalid. Did you mean:
 - something
 - soothing
 - seething
 - sheathing
 - southing
correct_word with "soemthing" took 1.868225s


In [9]:
corrector.correct_word('actop', d=1)

Word "actop" is invalid. Did you mean:
 - actor
 - atop
correct_word with "actop" took 0.000427s


## Conclusions

Building a simple approach for a spelling corrector is so easy performing a simple distance function. However, this is not suitable for a real environment as we've seen. For real world problems such as search engines, document editors... we would need much more complex and accurate models. These models would involve, for example, Complex Network Approach with graphs or Contextual Word Embeddings to use the context of the word to give a more appropiate prediction based on the sentence meaning.

However, not everything is black or white, there are middle points, and building a very simple probabilistic approach, we can get really nice results with high accuracy and fast performance, reducing the time required from 10s to 0.001s (with one edit step) and to 2s (with two edit steps).